# Data Analytics Project – Step-by-Step Analysis

This notebook walks through the entire data analytics pipeline interactively:
1. **Load** the sample retail sales dataset
2. **Inspect** the raw data
3. **Clean** the data (remove duplicates, handle missing values)
4. **Explore** the data with descriptive statistics
5. **Visualize** key patterns and trends
6. **Summarize** key business insights


## 0. Setup – Import Libraries

In [ ]:
# Standard library imports
import os
import sys

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Make plots appear inline in the notebook
%matplotlib inline

# Apply a consistent visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries imported successfully!')

---
## 1. Load the Dataset

In [ ]:
# Path to the sample dataset (relative to the notebook location)
DATA_PATH = os.path.join('data', 'sample_data.csv')

# Load the CSV file into a DataFrame
df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df_raw)} rows and {len(df_raw.columns)} columns.')
df_raw.head()

---
## 2. Inspect the Raw Data

In [ ]:
# Dataset dimensions
print(f'Shape: {df_raw.shape}')  # (rows, columns)

In [ ]:
# Column names and data types
df_raw.dtypes

In [ ]:
# Count of missing values per column
print('Missing values:')
df_raw.isnull().sum()

In [ ]:
# Number of duplicate rows
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

---
## 3. Data Cleaning

We will:
- Drop exact duplicate rows
- Parse the `date` column to datetime
- Fill numeric NaN values with the column **median**
- Fill categorical NaN values with `'Unknown'`
- Add derived columns: `year`, `month`, `month_label`

In [ ]:
# Step 1: Remove duplicate rows
df = df_raw.drop_duplicates()
print(f'Removed {len(df_raw) - len(df)} duplicate row(s). Remaining: {len(df)}')

In [ ]:
# Step 2: Parse the 'date' column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(df['date'].dtype)

In [ ]:
# Step 3: Fill missing numeric values with the column median
numeric_cols = ['quantity', 'unit_price', 'sales', 'discount', 'profit']
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  '{col}': filled NaN(s) with median = {median_val:.2f}")

In [ ]:
# Step 4: Fill missing categorical values with 'Unknown'
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna('Unknown')
        print(f"  '{col}': filled NaN(s) with 'Unknown'")

In [ ]:
# Step 5: Add derived date columns
df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['month_label'] = df['date'].dt.strftime('%b')

# Reset index for clean row numbering
df = df.reset_index(drop=True)

print(f'Clean dataset shape: {df.shape}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')
df.head()

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 Descriptive statistics for numeric columns
df.describe().round(2)

In [ ]:
# 4.2 Total sales and average profit by product category
by_category = (
    df.groupby('category')
    .agg(total_sales=('sales', 'sum'),
         avg_profit=('profit', 'mean'),
         order_count=('order_id', 'count'))
    .round(2)
    .sort_values('total_sales', ascending=False)
)
by_category

In [ ]:
# 4.3 Total sales and profit by region
by_region = (
    df.groupby('region')
    .agg(total_sales=('sales', 'sum'),
         total_profit=('profit', 'sum'),
         order_count=('order_id', 'count'))
    .round(2)
    .sort_values('total_sales', ascending=False)
)
by_region

In [ ]:
# 4.4 Monthly revenue and profit trends
monthly = (
    df.groupby(['year', 'month', 'month_label'])
    .agg(total_sales=('sales', 'sum'),
         total_profit=('profit', 'sum'))
    .round(2)
    .reset_index()
    .sort_values(['year', 'month'])
)
monthly

In [ ]:
# 4.5 Top 5 products by total sales
top_products = (
    df.groupby('product')
    .agg(total_sales=('sales', 'sum'),
         total_profit=('profit', 'sum'),
         units_sold=('quantity', 'sum'))
    .round(2)
    .sort_values('total_sales', ascending=False)
    .head(5)
)
top_products

In [ ]:
# 4.6 Correlation matrix for numeric columns
corr = df.select_dtypes(include='number').corr().round(3)
corr

---
## 5. Visualizations

In [ ]:
# 5.1 Bar chart – Total Sales by Category
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('muted', len(by_category))
bars = ax.bar(by_category.index, by_category['total_sales'], color=colors)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 50,
            f'${h:,.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Total Sales by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Total Sales ($)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Line plot – Monthly Sales & Profit Trend
monthly['period'] = monthly['month_label'] + ' ' + monthly['year'].astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(monthly['period'], monthly['total_sales'], marker='o',
        linewidth=2, label='Total Sales', color='#2196F3')
ax.plot(monthly['period'], monthly['total_profit'], marker='s',
        linewidth=2, linestyle='--', label='Total Profit', color='#4CAF50')

ax.set_title('Monthly Sales & Profit Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Amount ($)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Histogram – Distribution of Sales Amounts
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x='sales', bins=30, kde=True, color='#5C6BC0', ax=ax)

mean_val   = df['sales'].mean()
median_val = df['sales'].median()
ax.axvline(mean_val,   color='red',    linestyle='--', linewidth=1.5,
           label=f'Mean: ${mean_val:,.2f}')
ax.axvline(median_val, color='orange', linestyle='-.', linewidth=1.5,
           label=f'Median: ${median_val:,.2f}')

ax.set_title('Distribution of Sales Amounts', fontsize=14, fontweight='bold')
ax.set_xlabel('Sales ($)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Heatmap – Correlation Matrix
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (Numeric Variables)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Horizontal bar chart – Total Sales by Region
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('pastel', len(by_region))
bars = ax.barh(by_region.index, by_region['total_sales'], color=colors)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 50, bar.get_y() + bar.get_height() / 2,
            f'${w:,.0f}', va='center', fontsize=9)

ax.set_title('Total Sales by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)', fontsize=11)
ax.set_ylabel('Region', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 6. Key Insights

Based on the analysis:

In [ ]:
best_cat      = by_category['total_sales'].idxmax()
best_cat_val  = by_category.loc[best_cat, 'total_sales']

best_reg      = by_region['total_sales'].idxmax()
best_reg_val  = by_region.loc[best_reg, 'total_sales']

best_prod     = top_products.index[0]
best_prod_val = top_products.iloc[0]['total_sales']

avg_sales  = df['sales'].mean()
max_sale   = df['sales'].max()

print('=' * 60)
print('          KEY INSIGHTS FROM THE ANALYSIS')
print('=' * 60)
print(f'  Top category by sales : {best_cat} (${best_cat_val:,.2f})')
print(f'  Top region by sales   : {best_reg} (${best_reg_val:,.2f})')
print(f'  Best-selling product  : {best_prod} (${best_prod_val:,.2f})')
print(f'  Average order sales   : ${avg_sales:,.2f}')
print(f'  Highest single sale   : ${max_sale:,.2f}')
print('=' * 60)